# CNN Model — PyTorch (Rice Disease Dataset)

**Run all cells top-to-bottom (or use Kernel → Restart & Run All).**

| Cell | Content |
|------|---------|
| 1 | Imports & Device Setup |
| 2 | Configuration |
| 3 | Data Loading |
| 4 | Shared Utilities |
| 5 | **CNN** (Baseline) |
| 6 | Final Summary |


In [1]:
import os, copy, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'Device      : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'CUDA        : {torch.version.cuda}')
    print(f'Total VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')


PyTorch     : 2.5.1+cu121
Torchvision : 0.20.1+cu121
Device      : cuda
GPU         : NVIDIA RTX 5000 Ada Generation
CUDA        : 12.1
Total VRAM  : 31.99 GB


In [ ]:
# -- Hyper-parameters (consistent with GPU environment config) ----------------
BATCH_SIZE  = 64
IMAGE_SIZE  = (224, 224)
EPOCHS      = 50
RANDOM_SEED = 42

DATA_DIR   = r'..\Rice_Disease_Dataset'
OUTPUT_DIR = 'Model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'Output directory : {os.path.abspath(OUTPUT_DIR)}')


Output directory : c:\Users\km1612\Documents\riceMobileNet\Final\Multi_source_dataset\Model


In [7]:
# -- Discover classes ---------------------------------------------------------
classes = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
num_classes  = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}
NUM_CLASSES  = num_classes

print('Classes:    ', classes)
print('num_classes:', num_classes)

# -- Build filepath / label dataframe -----------------------------------------
filepaths, labels = [], []

for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            filepaths.append(os.path.join(cls_dir, f))
            labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print('Total Images:', len(df))

# -- 80 / 10 / 10 stratified split (matches original Keras notebooks) ----------
df_train, df_temp = train_test_split(
    df,
    test_size=0.20,
    shuffle=True,
    stratify=df['label'],
    random_state=123
)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    shuffle=True,
    stratify=df_temp['label'],
    random_state=123
)

print('Train:', len(df_train))
print('Val:  ', len(df_val))
print('Test: ', len(df_test))
# -- Class weights (PyTorch port of compute_class_weight 'balanced') ----------
from sklearn.utils.class_weight import compute_class_weight

y_train_encoded = df_train['label'].map(class_to_idx).values
class_weights   = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train_encoded
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print('Class weights:')
for cls, w in zip(classes, class_weights):
    print(f'  {cls:<25} : {w:.4f}')


Classes:     ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scaled', 'narrow_brown_spot']
num_classes: 6
Total Images: 14758
Train: 11806
Val:   1476
Test:  1476
Class weights:
  bacterial_leaf_blight     : 1.0383
  brown_spot                : 0.8195
  healthy                   : 1.0938
  leaf_blast                : 0.7428
  leaf_scaled               : 0.9760
  narrow_brown_spot         : 1.8811


In [8]:
# =============================================================================
#  SHARED UTILITIES  (run once -- used by every model below)
# =============================================================================

# -- Custom Dataset -----------------------------------------------------------
class ImageDataFrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df      = df.reset_index(drop=True)
        self.transform = transform
        self.encoded = [class_to_idx[l] for l in self.df['label']]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.encoded[idx]


# -- Normalization constants --------------------------------------------------
IMAGENET_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
MINUSONE_NORM = transforms.Normalize([0.5,   0.5,   0.5  ], [0.5,   0.5,   0.5  ])
IDENTITY_NORM = transforms.Normalize([0.0,   0.0,   0.0  ], [1.0,   1.0,   1.0  ])


# -- Early Stopping -----------------------------------------------------------
class EarlyStopping:
    def __init__(self, patience=7):
        self.patience     = patience
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience
    def restore(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


# -- DataLoader factory -------------------------------------------------------
def make_loaders(train_t, val_t, batch_size=BATCH_SIZE):
    pin = (DEVICE.type == 'cuda')
    train_loader = DataLoader(
        ImageDataFrameDataset(df_train, train_t),
        batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=pin)
    val_loader = DataLoader(
        ImageDataFrameDataset(df_val, val_t),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    test_loader = DataLoader(
        ImageDataFrameDataset(df_test, val_t),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    return train_loader, val_loader, test_loader


# -- Keras-style progress bar -------------------------------------------------
def _bar(current, total, width=30):
    filled = int(width * current / total)
    return '=' * filled + '-' * (width - filled)


# -- Training Engine ----------------------------------------------------------
def train_model(model, model_name, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5, verbose=False
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    es     = EarlyStopping(patience=7)

    history = {k: [] for k in [
        'loss', 'accuracy', 'precision', 'recall', 'f1',
        'val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1'
    ]}

    n_batches = len(train_loader)

    print(f'\nModel  : {model_name}')
    print(f'Train batches : {n_batches}  |  Val batches : {len(val_loader)}')
    print(f'Device : {DEVICE}' + (
        f'  ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
    print('-' * 110)

    for epoch in range(1, EPOCHS + 1):
        # -- Train ---
        model.train()
        t0 = time.time()
        ep_loss = 0.0
        ep_preds, ep_labels = [], []

        for bi, (imgs, labels) in enumerate(train_loader, 1):
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                out = model(imgs)
                if isinstance(out, torchvision.models.inception.InceptionOutputs):
                    out = out.logits
                loss = criterion(out, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            ep_loss += loss.item() * imgs.size(0)
            ep_preds.extend(out.argmax(1).detach().cpu().numpy())
            ep_labels.extend(labels.detach().cpu().numpy())

            # In-epoch progress update
            print(f'\rEpoch {epoch}/{EPOCHS} {bi}/{n_batches} [{_bar(bi, n_batches)}]',
                  end='', flush=True)

        # -- Train metrics ---
        tr_loss = ep_loss / len(train_loader.dataset)
        tr_acc  = accuracy_score(ep_labels, ep_preds)
        tr_prec = precision_score(ep_labels, ep_preds, average='weighted', zero_division=0)
        tr_rec  = recall_score(ep_labels,   ep_preds, average='weighted', zero_division=0)
        tr_f1   = f1_score(ep_labels,       ep_preds, average='weighted', zero_division=0)

        # -- Validate ---
        model.eval()
        vl_loss = 0.0
        vl_preds, vl_labels = [], []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(DEVICE,   non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                    out = model(imgs)
                    if isinstance(out, torchvision.models.inception.InceptionOutputs):
                        out = out.logits
                    loss = criterion(out, labels)
                vl_loss += loss.item() * imgs.size(0)
                vl_preds.extend(out.argmax(1).cpu().numpy())
                vl_labels.extend(labels.cpu().numpy())

        vl_loss /= len(val_loader.dataset)
        vl_acc   = accuracy_score(vl_labels, vl_preds)
        vl_prec  = precision_score(vl_labels, vl_preds, average='weighted', zero_division=0)
        vl_rec   = recall_score(vl_labels,   vl_preds, average='weighted', zero_division=0)
        vl_f1    = f1_score(vl_labels,       vl_preds, average='weighted', zero_division=0)

        elapsed   = time.time() - t0
        step_time = elapsed / n_batches

        # Keras-style epoch summary line
        print(
            f'\rEpoch {epoch}/{EPOCHS} {n_batches}/{n_batches} [{"=" * 30}] - '
            f'{elapsed:.0f}s {step_time:.1f}s/step - '
            f'loss: {tr_loss:.4f} - accuracy: {tr_acc:.4f} - '
            f'precision: {tr_prec:.4f} - recall: {tr_rec:.4f} - f1: {tr_f1:.4f} - '
            f'val_loss: {vl_loss:.4f} - val_accuracy: {vl_acc:.4f} - '
            f'val_precision: {vl_prec:.4f} - val_recall: {vl_rec:.4f} - val_f1: {vl_f1:.4f}',
            flush=True
        )
        print()

        for key, val in [
            ('loss',          tr_loss), ('accuracy',      tr_acc),
            ('precision',     tr_prec), ('recall',        tr_rec),  ('f1',      tr_f1),
            ('val_loss',      vl_loss), ('val_accuracy',  vl_acc),
            ('val_precision', vl_prec), ('val_recall',    vl_rec),  ('val_f1',  vl_f1),
        ]:
            history[key].append(round(float(val), 6))

        scheduler.step(vl_loss)

        if es(vl_loss, model):
            print(f'  >> Early stopping at epoch {epoch}. '
                  f'Best val_loss = {es.best_loss:.4f}')
            break

    es.restore(model)
    return history


# -- Test Evaluation ----------------------------------------------------------
def evaluate_on_test(model, test_loader):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    model.eval()
    tl, tp, tg = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                out = model(imgs)
                if isinstance(out, torchvision.models.inception.InceptionOutputs):
                    out = out.logits
                loss = criterion(out, labels)
            tl += loss.item() * imgs.size(0)
            tp.extend(out.argmax(1).cpu().numpy())
            tg.extend(labels.cpu().numpy())
    tl   /= len(test_loader.dataset)
    acc   = accuracy_score(tg, tp)
    prec  = precision_score(tg, tp, average='weighted', zero_division=0)
    rec   = recall_score(tg,   tp, average='weighted', zero_division=0)
    f1    = f1_score(tg,       tp, average='weighted', zero_division=0)
    print(f'\n{"=" * 60}')
    print('  TEST EVALUATION')
    print(f'{"=" * 60}')
    print(f'  loss      : {tl:.4f}')
    print(f'  accuracy  : {acc:.4f}')
    print(f'  precision : {prec:.4f}')
    print(f'  recall    : {rec:.4f}')
    print(f'  f1_score  : {f1:.4f}')
    print(f'{"=" * 60}')
    print()
    print('Classification Report:')
    print(classification_report(tg, tp, target_names=classes, digits=4))


# -- Save utilities -----------------------------------------------------------
def save_weights(model, model_name):
    # Saves model.state_dict() -- reload with:
    #   model = BuildFn(NUM_CLASSES)
    #   model.load_state_dict(torch.load('Model/<name>_weights.pth'))
    #   model.eval()
    path = os.path.join(OUTPUT_DIR, f'{model_name}_weights.pth')
    torch.save(model.state_dict(), path)
    print(f'  Weights saved  ->  {path}')

def save_history(history, model_name):
    # Saves history dict as JSON (mirrors Keras history.history)
    path = os.path.join(OUTPUT_DIR, f'{model_name}_history.json')
    with open(path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f'  History saved  ->  {path}')


print('All shared utilities ready.')


All shared utilities ready.


In [9]:
# =============================================================================
#  MODEL 1 : CNN  (Baseline)
# =============================================================================
print('=' * 70)
print('  MODEL: CNN (Baseline)')
print('=' * 70)

# -- Transforms ---------------------------------------------------------------
# Keras original: rotation=25, width/height_shift=0.15, shear=0.1, fill='nearest'
# IDENTITY_NORM: ToTensor() already gives [0,1] -> mirrors Keras Rescaling(1/255)
cnn_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    IDENTITY_NORM,
])
cnn_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    IDENTITY_NORM,
])

cnn_train_loader, cnn_val_loader, cnn_test_loader = make_loaders(cnn_train_t, cnn_val_t)

# -- Architecture (exact PyTorch port of original Keras CNN) ------------------
# Keras: Conv(32,3)->ReLU->MaxPool  |  Conv(64,3)->ReLU->MaxPool
#        Conv(64,3)->ReLU->MaxPool  |  Flatten  |  Dense(64,relu)  |  Dense(n)
# Spatial flow for 224x224 input:
#   224 -[conv3]-> 222 -[pool2]-> 111
#   111 -[conv3]-> 109 -[pool2]->  54
#    54 -[conv3]->  52 -[pool2]->  26
#   Flatten: 64 * 26 * 26 = 43264
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, kernel_size=3), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 26 * 26, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

cnn_model = SimpleCNN(NUM_CLASSES).to(DEVICE)
print(f'Trainable params : {sum(p.numel() for p in cnn_model.parameters() if p.requires_grad):,}')

cnn_history = train_model(cnn_model, 'CNN', cnn_train_loader, cnn_val_loader)
evaluate_on_test(cnn_model, cnn_test_loader)

save_weights(cnn_model, 'CNN')
save_history(cnn_history, 'CNN')

del cnn_model, cnn_train_loader, cnn_val_loader, cnn_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nCNN done.\n')


  MODEL: CNN (Baseline)
Trainable params : 2,825,670

Model  : CNN
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 30s 0.2s/step - loss: 1.7951 - accuracy: 0.1893 - precision: 0.3006 - recall: 0.1893 - f1: 0.1375 - val_loss: 1.7669 - val_accuracy: 0.2053 - val_precision: 0.0988 - val_recall: 0.2053 - val_f1: 0.0863

Epoch 2/50 185/185 [==============================] - 29s 0.2s/step - loss: 1.7227 - accuracy: 0.2754 - precision: 0.4221 - recall: 0.2754 - f1: 0.2198 - val_loss: 1.6543 - val_accuracy: 0.4682 - val_precision: 0.4991 - val_recall: 0.4682 - val_f1: 0.4655

Epoch 3/50 185/185 [==============================] - 26s 0.1s/step - loss: 1.6194 - accuracy: 0.3756 - precision: 0.4527 - recall: 0.3756 - f1: 0.3636 - val_loss: 1.5571 - val_accuracy: 0.3936 - val_precision: 0.4735 -

In [10]:
# =============================================================================
#  TRAINING COMPLETE  -- all weights + histories saved under Model/
# =============================================================================
import glob as _glob

print('=' * 70)
print('  CNN TRAINING COMPLETE')
print('=' * 70)
print()
print('Saved files:')
for p in sorted(_glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    size_kb = os.path.getsize(p) / 1024
    print(f'  {p:<55}  ({size_kb:.1f} KB)')

print()
print('-- How to reload weights (example: CNN) --')
print("  model = SimpleCNN(NUM_CLASSES)")
print("  model.load_state_dict(torch.load('Model/CNN_weights.pth', map_location=DEVICE))")
print("  model.to(DEVICE).eval()")
print()
print('-- How to reload history (example: CNN) --')
print("  with open('Model/CNN_history.json') as f:")
print("      h = json.load(f)")
print("  # h['val_accuracy'], h['val_f1'], h['loss'], ...")


  CNN TRAINING COMPLETE

Saved files:
  Model\CNN_history.json                                   (7.5 KB)
  Model\CNN_weights.pth                                    (11041.9 KB)

-- How to reload weights (example: CNN) --
  model = SimpleCNN(NUM_CLASSES)
  model.load_state_dict(torch.load('Model/CNN_weights.pth', map_location=DEVICE))
  model.to(DEVICE).eval()

-- How to reload history (example: CNN) --
  with open('Model/CNN_history.json') as f:
      h = json.load(f)
  # h['val_accuracy'], h['val_f1'], h['loss'], ...
